In [1]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from torchvision.utils import make_grid
from torchvision.datasets import ImageFolder
from torchsummary import summary
from PIL import Image
import json

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms, models

from sklearn.metrics import confusion_matrix

%matplotlib inline

In [2]:
ds = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"
diseases  = os.listdir(ds)
print(len(diseases))  #38 classes
# print(diseases )

38


# t

In [3]:
# ============================================================
# Vanilla ProtoNet Baseline: Multi-Seed PlantVillage
# Compatible with UA-ProtoNet fixed episodes for Task 1.2
# Seeds: 42, 123, 456
# ============================================================

import os
import gc
import random
import json
import glob
import math
import contextlib
from collections import defaultdict
from dataclasses import dataclass
from typing import Optional, List, Dict, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from scipy.stats import t


# ============================================================
# Seed & Reproducibility
# ============================================================

def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def worker_init_fn(worker_id: int):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def autocast_context(device: str):
    if str(device).startswith("cuda"):
        try:
            return torch.amp.autocast(device_type="cuda")
        except AttributeError:
            return torch.cuda.amp.autocast()
    return contextlib.nullcontext()


def make_grad_scaler(use_cuda: bool):
    if not use_cuda:
        return None
    try:
        return torch.amp.GradScaler(device="cuda")
    except AttributeError:
        return torch.cuda.amp.GradScaler()


# ============================================================
# Configuration
# ============================================================

@dataclass
class Config:
    data_root: str = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"

    save_dir: str = "baseline_runs"
    episode_dir: str = "."
    results_dir: str = "baseline_runs"

    split_seed: int = 42
    seeds: Tuple[int, ...] = (42, 123, 456)

    image_size: int = 224
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    batch_size: int = 1
    
    # CRITICAL FIX: Set to 0 to prevent Kaggle NFS deadlocks between seeds.
    # Multiprocessing workers frequently fail to release file handles on Kaggle,
    # causing the notebook to hang indefinitely when starting the next seed.
    num_workers: int = 0

    lr: float = 1e-3
    weight_decay: float = 1e-4
    epochs: int = 50
    patience: int = 10

    train_episodes: int = 600
    val_episodes: int = 300
    test_episodes: int = 600

    n_way: int = 5
    k_shot: int = 5
    q_query: int = 15

    use_projection: bool = False
    embed_dim: int = 256
    normalize_embeddings: bool = False
    squared_distance: bool = True

    use_train_flip: bool = True

    eval_seed: int = 20260725

    direct_checkpoint_paths: Tuple[str, ...] = (
        "/kaggle/input/models/tanvirakon/bse-42/pytorch/default/1/vanilla_baseline_seed42.pth",
    )


CFG = Config()


# ============================================================
# Vanilla EfficientNetB0 ProtoNet Baseline
# ============================================================

class EfficientNetB0VanillaBaseline(nn.Module):
    def __init__(
        self,
        pretrained: bool = True,
        use_projection: bool = False,
        embed_dim: int = 256,
        normalize_embeddings: bool = False,
    ):
        super().__init__()

        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        effnet = models.efficientnet_b0(weights=weights)

        self.features = effnet.features
        self.pooling = nn.AdaptiveAvgPool2d(1)

        self.use_projection = use_projection
        self.normalize_embeddings = normalize_embeddings

        if self.use_projection:
            self.proj = nn.Linear(1280, embed_dim)

    def forward(self, x):
        x = self.features(x)
        x = self.pooling(x)
        x = torch.flatten(x, 1)

        if self.use_projection:
            x = self.proj(x)

        if self.normalize_embeddings:
            x = F.normalize(x, dim=1)

        return x


# ============================================================
# ProtoNet Utilities
# ============================================================

def compute_prototypes(embeddings, labels, n_way):
    embeddings = embeddings.float()
    return torch.stack([
        embeddings[labels == c].mean(dim=0)
        for c in range(n_way)
    ])


def euclidean_logits(query_embeddings, prototypes, squared: bool = True):
    query_embeddings = query_embeddings.float()
    prototypes = prototypes.float()

    dist = torch.cdist(query_embeddings, prototypes, p=2)

    if squared:
        dist = dist.pow(2)

    return -dist


# ============================================================
# Dataset Helpers
# ============================================================

class ImageFolderWithIndex(datasets.ImageFolder):
    def __getitem__(self, index: int):
        image, label = super().__getitem__(index)
        return image, label, index


def build_class_to_indices(dataset) -> Dict[int, List[int]]:
    class_to_indices = defaultdict(list)
    for idx, target in enumerate(dataset.targets):
        class_to_indices[target].append(idx)
    return dict(class_to_indices)


def get_pv_class_split(num_classes: int, split_seed: int = 42):
    if num_classes < 38:
        raise ValueError(
            f"Expected at least 38 PlantVillage classes for 23/8/7 split, found {num_classes}."
        )

    all_classes = list(range(num_classes))
    rng = random.Random(split_seed)
    rng.shuffle(all_classes)

    train_classes = all_classes[:23]
    val_classes = all_classes[23:31]
    test_classes = all_classes[31:38]

    return train_classes, val_classes, test_classes


# ============================================================
# Episodic Datasets
# ============================================================

class FewShotEpisodeDataset(Dataset):
    def __init__(
        self,
        base_dataset,
        valid_classes,
        class_to_indices,
        n_way,
        k_shot,
        q_query,
        episodes
    ):
        self.base_dataset = base_dataset
        self.valid_classes = valid_classes
        self.class_to_indices = class_to_indices
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query
        self.episodes = episodes

    def __len__(self):
        return self.episodes

    def __getitem__(self, _):
        sampled_classes = random.sample(self.valid_classes, self.n_way)

        support_x, support_y = [], []
        query_x, query_y = [], []

        for mapped_label, cls in enumerate(sampled_classes):
            available = self.class_to_indices[cls]
            needed = self.k_shot + self.q_query

            if len(available) < needed:
                raise ValueError(
                    f"Class {cls} has only {len(available)} images, but needs {needed}."
                )

            selected = random.sample(available, needed)

            for i in selected[:self.k_shot]:
                img, _, _ = self.base_dataset[i]
                support_x.append(img)
                support_y.append(mapped_label)

            for i in selected[self.k_shot:self.k_shot + self.q_query]:
                img, _, _ = self.base_dataset[i]
                query_x.append(img)
                query_y.append(mapped_label)

        return (
            torch.stack(support_x),
            torch.tensor(support_y, dtype=torch.long),
            torch.stack(query_x),
            torch.tensor(query_y, dtype=torch.long)
        )


class FixedEpisodeDataset(Dataset):
    def __init__(self, base_dataset, episodes: List[dict]):
        self.base_dataset = base_dataset
        self.episodes = episodes

    def __len__(self):
        return len(self.episodes)

    def __getitem__(self, idx):
        ep = self.episodes[idx]

        support_x = torch.stack([
            self.base_dataset[int(i)][0] for i in ep["support_indices"]
        ])
        query_x = torch.stack([
            self.base_dataset[int(i)][0] for i in ep["query_indices"]
        ])

        support_y = torch.tensor(ep["support_labels"], dtype=torch.long)
        query_y = torch.tensor(ep["query_labels"], dtype=torch.long)

        return support_x, support_y, query_x, query_y


# ============================================================
# Fixed Episode Generation / Loading
# ============================================================

def load_or_create_fixed_episodes_ua_format(
    path: str,
    split_name: str,
    valid_classes: List[int],
    class_to_indices: Dict[int, List[int]],
    n_way: int,
    k_shot: int,
    q_query: int,
    episodes: int,
    rng_seed: int
) -> List[dict]:
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)

    if os.path.exists(path):
        print(f"Loading fixed {split_name} episodes from: {path}")

        with open(path, "r") as f:
            data = json.load(f)

        if not isinstance(data, list) or len(data) == 0:
            raise RuntimeError(
                f"{path} exists but is not a non-empty UA-format episode list. "
                f"Delete it or point episode_dir to the UA-ProtoNet episode folder."
            )

        required_keys = {
            "episode_id",
            "support_indices",
            "support_labels",
            "query_indices",
            "query_labels"
        }

        if not required_keys.issubset(data[0].keys()):
            raise RuntimeError(
                f"{path} exists but is not UA-ProtoNet episode format. "
                f"Delete it or point episode_dir to the UA-ProtoNet episode folder."
            )

        return data

    print(f"Creating fixed {split_name} episodes at: {path}")
    rng = random.Random(rng_seed)

    episode_list = []

    for episode_id in range(episodes):
        sampled_classes = rng.sample(list(valid_classes), n_way)

        support_indices = []
        support_labels = []
        query_indices = []
        query_labels = []

        for mapped_label, cls in enumerate(sampled_classes):
            available = class_to_indices[cls]
            needed = k_shot + q_query

            if len(available) < needed:
                raise ValueError(
                    f"Class {cls} has only {len(available)} images, but needs {needed}."
                )

            selected = rng.sample(available, needed)

            support_indices.extend(selected[:k_shot])
            support_labels.extend([mapped_label] * k_shot)

            query_indices.extend(selected[k_shot:k_shot + q_query])
            query_labels.extend([mapped_label] * q_query)

        episode_list.append({
            "episode_id": episode_id,
            "classes": sampled_classes,
            "support_indices": support_indices,
            "support_labels": support_labels,
            "query_indices": query_indices,
            "query_labels": query_labels
        })

    with open(path, "w") as f:
        json.dump(episode_list, f)

    return episode_list


# ============================================================
# Safe Checkpoint Finder
# ============================================================

def find_checkpoint(run_seed: int, cfg: Config) -> Optional[str]:
    candidate_names = [
        f"baseline_seed_{run_seed}.pth",
        f"baseline_seed_{run_seed}.pt",
        f"baseline_seed_{run_seed}.ckpt",
        f"vanilla_baseline_seed_{run_seed}.pth",
        f"vanilla_baseline_seed{run_seed}.pth",
        f"seed_{run_seed}.pth",
        f"seed{run_seed}.pth",
    ]

    if run_seed == 42:
        candidate_names += [
            "seed42.pth",
            "baseline_seed42.pth",
            "vanilla_baseline_seed42.pth",
            "vill_baseline.pth",
            "vill_baseline_tst_score.pth",
        ]

    seen = set()
    unique_names = []
    for name in candidate_names:
        if name not in seen:
            seen.add(name)
            unique_names.append(name)
    candidate_names = unique_names
    candidate_set = set(candidate_names)

    # 1. Direct known paths
    for path in cfg.direct_checkpoint_paths:
        if os.path.isfile(path):
            fname = os.path.basename(path).lower()
            if f"seed_{run_seed}" in fname or f"seed{run_seed}" in fname or (run_seed == 42 and "42" in fname):
                return path

    # 2. Current directory, baseline_runs, Kaggle working directory
    search_bases = [".", cfg.save_dir, "/kaggle/working"]
    for base in search_bases:
        if not base: continue
        for name in candidate_names:
            path = os.path.join(base, name)
            if os.path.isfile(path):
                return path

    # 3. Safe Kaggle input walk (prevents deep glob.glob NFS hangs)
    base_input = "/kaggle/input"
    if os.path.isdir(base_input):
        prune_tokens = (
            "plantvillage-dataset",
            "plantdoc-test-train-merged",
            "abdallahalidev/plantvillage",
            "tanvirakon/plantdoc",
            "color", 
        )

        for root, dirs, files in os.walk(base_input):
            root_lower = root.lower()
            if any(token in root_lower for token in prune_tokens):
                dirs[:] = []
                continue
            
            dirs[:] = [d for d in dirs if not d.startswith(".")]

            for file in files:
                if file in candidate_set:
                    full_path = os.path.join(root, file)
                    if os.path.isfile(full_path):
                        return full_path

    return None


def load_checkpoint_compat(model, path: str, device: str):
    try:
        state = torch.load(path, map_location=device, weights_only=True)
    except Exception:
        state = torch.load(path, map_location=device)

    if isinstance(state, nn.Module):
        state = state.state_dict()

    if isinstance(state, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            if key in state and isinstance(state[key], dict):
                state = state[key]
                break

        state = {
            k.replace("module.", ""): v
            for k, v in state.items()
        }

    missing, unexpected = model.load_state_dict(state, strict=False)

    if missing:
        raise RuntimeError(
            f"Checkpoint is missing required keys for this vanilla baseline:\n{missing}"
        )

    if unexpected:
        print(f"WARNING: unexpected checkpoint keys were ignored: {unexpected}")


# ============================================================
# Training Loop
# ============================================================

def train_fsl(model, train_loader, val_loader, cfg: Config, run_seed: int):
    use_cuda = str(cfg.device).startswith("cuda")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay
    )

    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=15,
        gamma=0.5
    )

    scaler = make_grad_scaler(use_cuda)

    best_val_acc = -1.0
    epochs_no_improve = 0

    os.makedirs(cfg.save_dir, exist_ok=True)
    save_filename = os.path.join(cfg.save_dir, f"baseline_seed_{run_seed}.pth")

    print(f"[Baseline Seed {run_seed}] Starting training from scratch for {cfg.epochs} max epochs...", flush=True)

    for epoch in range(cfg.epochs):
        model.train()

        train_loss = 0.0
        train_acc = 0.0

        for ep_idx, (support_x, support_y, query_x, query_y) in enumerate(train_loader):
            support_x = support_x.squeeze(0).to(cfg.device, non_blocking=True)
            support_y = support_y.squeeze(0).to(cfg.device, non_blocking=True)
            query_x = query_x.squeeze(0).to(cfg.device, non_blocking=True)
            query_y = query_y.squeeze(0).to(cfg.device, non_blocking=True)

            optimizer.zero_grad()

            with autocast_context(cfg.device):
                all_emb = model(torch.cat([support_x, query_x], dim=0))

            all_emb = all_emb.float()

            n_support = support_x.size(0)
            support_emb = all_emb[:n_support]
            query_emb = all_emb[n_support:]

            prototypes = compute_prototypes(support_emb, support_y, cfg.n_way)
            logits = euclidean_logits(query_emb, prototypes, cfg.squared_distance)

            loss = F.cross_entropy(logits, query_y)

            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

            train_loss += loss.item()
            train_acc += (logits.argmax(1) == query_y).float().mean().item()

            if (ep_idx + 1) % 200 == 0:
                print(f"   [Epoch {epoch + 1:02d}/{cfg.epochs:02d}] Train Ep {ep_idx + 1}/{len(train_loader)}", flush=True)

        train_loss /= len(train_loader)
        train_acc /= len(train_loader)

        model.eval()
        val_acc = 0.0

        with torch.no_grad():
            for support_x, support_y, query_x, query_y in val_loader:
                support_x = support_x.squeeze(0).to(cfg.device, non_blocking=True)
                support_y = support_y.squeeze(0).to(cfg.device, non_blocking=True)
                query_x = query_x.squeeze(0).to(cfg.device, non_blocking=True)
                query_y = query_y.squeeze(0).to(cfg.device, non_blocking=True)

                with autocast_context(cfg.device):
                    all_emb = model(torch.cat([support_x, query_x], dim=0))

                all_emb = all_emb.float()

                n_support = support_x.size(0)
                support_emb = all_emb[:n_support]
                query_emb = all_emb[n_support:]

                prototypes = compute_prototypes(support_emb, support_y, cfg.n_way)
                logits = euclidean_logits(query_emb, prototypes, cfg.squared_distance)

                val_acc += (logits.argmax(1) == query_y).float().mean().item()

        val_acc /= len(val_loader)

        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        print(
            f"[Baseline Seed {run_seed:4d}] Epoch {epoch+1:02d}/{cfg.epochs} "
            f"[LR: {current_lr:.6f}] | "
            f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), save_filename)
            print(f"  --> Best baseline model saved to '{save_filename}'. Val Acc: {best_val_acc:.4f}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= cfg.patience:
                print(f"\nEarly stopping at epoch {epoch+1}. Best Val Acc: {best_val_acc:.4f}")
                break

    load_checkpoint_compat(model, save_filename, cfg.device)
    print(f"\nBaseline seed {run_seed} training complete. Best Val Acc: {best_val_acc:.4f}")

    return model, best_val_acc


# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate_baseline_pv(model, test_loader, cfg: Config, run_seed: int):
    model.eval()

    records = []

    print(f"[Baseline Seed {run_seed}] Evaluating fixed PlantVillage test episodes...", flush=True)

    for ep_idx, (support_x, support_y, query_x, query_y) in enumerate(test_loader):
        support_x = support_x.squeeze(0).to(cfg.device, non_blocking=True)
        support_y = support_y.squeeze(0).to(cfg.device, non_blocking=True)
        query_x = query_x.squeeze(0).to(cfg.device, non_blocking=True)
        query_y = query_y.squeeze(0).to(cfg.device, non_blocking=True)

        with autocast_context(cfg.device):
            all_emb = model(torch.cat([support_x, query_x], dim=0))

        all_emb = all_emb.float()

        n_support = support_x.size(0)
        support_emb = all_emb[:n_support]
        query_emb = all_emb[n_support:]

        prototypes = compute_prototypes(support_emb, support_y, cfg.n_way)
        logits = euclidean_logits(query_emb, prototypes, cfg.squared_distance)

        acc = (logits.argmax(1) == query_y).float().mean().item()

        records.append({
            "seed": run_seed,
            "episode_id": ep_idx,
            "det_acc": acc
        })

        if (ep_idx + 1) % 100 == 0:
            print(f"  Episode {ep_idx + 1}/{len(test_loader)}", flush=True)

    df = pd.DataFrame(records)

    os.makedirs(cfg.results_dir, exist_ok=True)
    csv_path = os.path.join(cfg.results_dir, f"baseline_pv_episode_metrics_seed_{run_seed}.csv")
    df.to_csv(csv_path, index=False)

    mean_acc = float(df["det_acc"].mean())
    print(f"[Baseline Seed {run_seed}] Test Acc: {mean_acc * 100:.2f}%", flush=True)
    print(f"Saved episode metrics: {csv_path}", flush=True)

    return mean_acc


# ============================================================
# Student's t Confidence Interval
# ============================================================

def compute_student_t_ci(values: List[float]) -> dict:
    vals = np.asarray(values, dtype=np.float64)
    n = len(vals)

    if n == 0:
        return {
            "mean": float("nan"),
            "ci": float("nan"),
            "std": float("nan"),
            "tcrit": float("nan"),
            "df": 0
        }

    mean = float(np.mean(vals))

    if n == 1:
        return {
            "mean": mean,
            "ci": 0.0,
            "std": 0.0,
            "tcrit": float("nan"),
            "df": 0
        }

    std = float(np.std(vals, ddof=1))
    df = n - 1

    tcrit = float(t.ppf(0.975, df))
    ci = tcrit * (std / math.sqrt(n))

    return {
        "mean": mean,
        "ci": ci,
        "std": std,
        "tcrit": tcrit,
        "df": df
    }


# ============================================================
# Summary Reporting
# ============================================================

def summarize_baseline_pv(all_metrics: List[dict], cfg: Config):
    os.makedirs(cfg.results_dir, exist_ok=True)

    seed_csv = os.path.join(cfg.results_dir, "baseline_pv_phase1_seed_metrics.csv")

    seed_rows = []
    for m in all_metrics:
        best_val = m.get("best_val_acc", None)

        seed_rows.append({
            "seed": m["seed"],
            "best_val_acc_pct": "" if best_val is None else best_val * 100.0,
            "det_test_acc_pct": m["test_acc"] * 100.0
        })

    pd.DataFrame(seed_rows).to_csv(seed_csv, index=False)
    print(f"Saved baseline seed-level metrics: {seed_csv}")

    test_acc_values = [m["test_acc"] * 100.0 for m in all_metrics]
    stats = compute_student_t_ci(test_acc_values)

    summary_csv = os.path.join(cfg.results_dir, "baseline_pv_phase1_summary.csv")

    summary_df = pd.DataFrame([
        {
            "metric": "Deterministic Test Accuracy (%)",
            "mean": stats["mean"],
            "student_t_95_ci_half_width": stats["ci"],
            "std": stats["std"],
            "df": stats["df"],
            "tcrit": stats["tcrit"],
            "seed_values": ";".join(f"{v:.6f}" for v in test_acc_values)
        }
    ])

    summary_df.to_csv(summary_csv, index=False)
    print(f"Saved baseline summary metrics: {summary_csv}")

    print("\n" + "=" * 90)
    print("  BASELINE PLANTVILLAGE MULTI-SEED SUMMARY")
    print("=" * 90)

    for m in all_metrics:
        best_val = m.get("best_val_acc", None)
        best_val_str = "NA" if best_val is None else f"{best_val * 100:.2f}%"

        print(
            f"  Seed {m['seed']:5d} | "
            f"Best Val: {best_val_str} | "
            f"Det Test: {m['test_acc'] * 100:.2f}%"
        )

    print("-" * 90)

    print(
        f"  Deterministic Test Accuracy (%): "
        f"{stats['mean']:.2f} ± {stats['ci']:.2f} "
        f"(95% CI, Student's t, df={stats['df']}, tcrit={stats['tcrit']:.3f}) | "
        f"SD={stats['std']:.2f}"
    )

    print("=" * 90)


# ============================================================
# Main Execution
# ============================================================

if __name__ == "__main__":
    print("=" * 90)
    print("  Vanilla ProtoNet Baseline: Multi-Seed PlantVillage Evaluation")
    print(f"  Seeds: {CFG.seeds}")
    print(f"  Fixed class split seed: {CFG.split_seed}")
    print(f"  Protocol: {CFG.n_way}-way {CFG.k_shot}-shot {CFG.q_query}-query")
    print("=" * 90)

    os.makedirs(CFG.save_dir, exist_ok=True)
    os.makedirs(CFG.results_dir, exist_ok=True)

    with open(os.path.join(CFG.results_dir, "baseline_pv_config.json"), "w") as f:
        json.dump(vars(CFG), f, indent=2)

    if CFG.use_train_flip:
        train_tfms = transforms.Compose([
            transforms.Resize((CFG.image_size, CFG.image_size)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:
        train_tfms = transforms.Compose([
            transforms.Resize((CFG.image_size, CFG.image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    val_tfms = transforms.Compose([
        transforms.Resize((CFG.image_size, CFG.image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print(f"Loading dataset from: {CFG.data_root}")

    full_ds = ImageFolderWithIndex(CFG.data_root, transform=train_tfms)
    val_ds = ImageFolderWithIndex(CFG.data_root, transform=val_tfms)

    class_to_indices = build_class_to_indices(full_ds)

    train_classes, val_classes, test_classes = get_pv_class_split(
        num_classes=len(full_ds.classes),
        split_seed=CFG.split_seed
    )

    print(
        f"Fixed Split (Seed {CFG.split_seed}): "
        f"{len(train_classes)} Train | "
        f"{len(val_classes)} Val | "
        f"{len(test_classes)} Test"
    )

    val_episodes_json = load_or_create_fixed_episodes_ua_format(
        path=os.path.join(
            CFG.episode_dir,
            f"pv_val_fixed_episodes_split{CFG.split_seed}_"
            f"n{CFG.n_way}_k{CFG.k_shot}_q{CFG.q_query}_ep{CFG.val_episodes}.json"
        ),
        split_name="val",
        valid_classes=val_classes,
        class_to_indices=class_to_indices,
        n_way=CFG.n_way,
        k_shot=CFG.k_shot,
        q_query=CFG.q_query,
        episodes=CFG.val_episodes,
        rng_seed=CFG.split_seed + 100001
    )

    test_episodes_json = load_or_create_fixed_episodes_ua_format(
        path=os.path.join(
            CFG.episode_dir,
            f"pv_test_fixed_episodes_split{CFG.split_seed}_"
            f"n{CFG.n_way}_k{CFG.k_shot}_q{CFG.q_query}_ep{CFG.test_episodes}.json"
        ),
        split_name="test",
        valid_classes=test_classes,
        class_to_indices=class_to_indices,
        n_way=CFG.n_way,
        k_shot=CFG.k_shot,
        q_query=CFG.q_query,
        episodes=CFG.test_episodes,
        rng_seed=CFG.split_seed + 200002
    )

    use_cuda = str(CFG.device).startswith("cuda")

    loader_kwargs = dict(
        batch_size=CFG.batch_size,
        num_workers=CFG.num_workers,
        pin_memory=use_cuda,
        persistent_workers=False,
        worker_init_fn=worker_init_fn
    )

    all_seed_metrics = []

    for seed_idx, run_seed in enumerate(CFG.seeds):
        print("\n" + "-" * 90, flush=True)
        print(f"   BASELINE SEED {run_seed} ({seed_idx + 1}/{len(CFG.seeds)})", flush=True)
        print("-" * 90, flush=True)
        print(f"   >> Setting seed and building datasets...", flush=True)

        set_seed(run_seed)

        train_task_ds = FewShotEpisodeDataset(
            base_dataset=full_ds,
            valid_classes=train_classes,
            class_to_indices=class_to_indices,
            n_way=CFG.n_way,
            k_shot=CFG.k_shot,
            q_query=CFG.q_query,
            episodes=CFG.train_episodes
        )

        val_task_ds = FixedEpisodeDataset(
            base_dataset=val_ds,
            episodes=val_episodes_json
        )

        test_task_ds = FixedEpisodeDataset(
            base_dataset=val_ds,
            episodes=test_episodes_json
        )

        train_generator = torch.Generator()
        train_generator.manual_seed(run_seed)

        train_loader = DataLoader(
            train_task_ds,
            shuffle=True,
            generator=train_generator,
            **loader_kwargs
        )

        val_loader = DataLoader(
            val_task_ds,
            shuffle=False,
            **loader_kwargs
        )

        test_loader = DataLoader(
            test_task_ds,
            shuffle=False,
            **loader_kwargs
        )

        print(f"   >> DataLoaders ready. Searching for checkpoint...", flush=True)
        checkpoint_path = find_checkpoint(run_seed, CFG)

        if checkpoint_path is not None:
            print(f"[SKIP TRAINING] Found existing baseline checkpoint for seed {run_seed}:", flush=True)
            print(f"  {checkpoint_path}", flush=True)
            model = EfficientNetB0VanillaBaseline(
                pretrained=False,
                use_projection=CFG.use_projection,
                embed_dim=CFG.embed_dim,
                normalize_embeddings=CFG.normalize_embeddings
            ).to(CFG.device)
            load_checkpoint_compat(model, checkpoint_path, CFG.device)
            best_val_acc = None
        else:
            model = EfficientNetB0VanillaBaseline(
                pretrained=True,
                use_projection=CFG.use_projection,
                embed_dim=CFG.embed_dim,
                normalize_embeddings=CFG.normalize_embeddings
            ).to(CFG.device)
            model, best_val_acc = train_fsl(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                cfg=CFG,
                run_seed=run_seed
            )

        set_seed(CFG.eval_seed)

        test_acc = evaluate_baseline_pv(
            model=model,
            test_loader=test_loader,
            cfg=CFG,
            run_seed=run_seed
        )

        all_seed_metrics.append({
            "seed": run_seed,
            "best_val_acc": best_val_acc,
            "test_acc": test_acc
        })

        # Explicit cleanup to prevent memory/NFS leaks between seeds
        del model
        del train_loader
        del val_loader
        del test_loader
        
        if use_cuda:
            torch.cuda.empty_cache()

        gc.collect()

    summarize_baseline_pv(all_seed_metrics, CFG)

  Vanilla ProtoNet Baseline: Multi-Seed PlantVillage Evaluation
  Seeds: (42, 123, 456)
  Fixed class split seed: 42
  Protocol: 5-way 5-shot 15-query
Loading dataset from: /kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color
Fixed Split (Seed 42): 23 Train | 8 Val | 7 Test
Creating fixed val episodes at: ./pv_val_fixed_episodes_split42_n5_k5_q15_ep300.json
Creating fixed test episodes at: ./pv_test_fixed_episodes_split42_n5_k5_q15_ep600.json

------------------------------------------------------------------------------------------
   BASELINE SEED 42 (1/3)
------------------------------------------------------------------------------------------
   >> Setting seed and building datasets...
   >> DataLoaders ready. Searching for checkpoint...
[SKIP TRAINING] Found existing baseline checkpoint for seed 42:
  /kaggle/input/models/tanvirakon/bse-42/pytorch/default/1/vanilla_baseline_seed42.pth
[Baseline Seed 42] Evaluating fixed PlantVillage test episodes...
  Episode 100/600


100%|██████████| 20.5M/20.5M [00:00<00:00, 129MB/s] 


[Baseline Seed 123] Starting training from scratch for 50 max epochs...
   [Epoch 01/50] Train Ep 200/600
   [Epoch 01/50] Train Ep 400/600
   [Epoch 01/50] Train Ep 600/600
[Baseline Seed  123] Epoch 01/50 [LR: 0.001000] | Train Loss: 0.1778 Acc: 0.9690 | Val Acc: 0.9667
  --> Best baseline model saved to 'baseline_runs/baseline_seed_123.pth'. Val Acc: 0.9667
   [Epoch 02/50] Train Ep 200/600
   [Epoch 02/50] Train Ep 400/600
   [Epoch 02/50] Train Ep 600/600
[Baseline Seed  123] Epoch 02/50 [LR: 0.001000] | Train Loss: 0.0284 Acc: 0.9908 | Val Acc: 0.9707
  --> Best baseline model saved to 'baseline_runs/baseline_seed_123.pth'. Val Acc: 0.9707
   [Epoch 03/50] Train Ep 200/600
   [Epoch 03/50] Train Ep 400/600
   [Epoch 03/50] Train Ep 600/600
[Baseline Seed  123] Epoch 03/50 [LR: 0.001000] | Train Loss: 0.0249 Acc: 0.9924 | Val Acc: 0.9739
  --> Best baseline model saved to 'baseline_runs/baseline_seed_123.pth'. Val Acc: 0.9739
   [Epoch 04/50] Train Ep 200/600
   [Epoch 04/50] Trai

# a

# b

In [4]:
# ============================================================
# Vanilla ProtoNet Baseline: Multi-Seed PlantDoc Evaluation
# Uses the exact same fixed episodes as UA-ProtoNet.
# Seeds: 42, 123, 456
# ============================================================

import os
import gc
import json
import math
import random
import hashlib
import contextlib
from collections import defaultdict
from dataclasses import dataclass
from typing import Optional, List, Dict, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from scipy.stats import t

# ============================================================
# Configuration
# ============================================================

@dataclass
class Config:
    plantvillage_root: str = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"
    plantdoc_root: str = "/kaggle/input/datasets/tanvirakon/plantdoc-test-train-merged/combined"

    episode_dir: str = "."
    results_dir: str = "baseline_runs"

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    image_size: int = 224

    batch_size: int = 1
    
    # CRITICAL FIX: Set to 0 to prevent Kaggle NFS deadlocks between seeds.
    num_workers: int = 0

    n_way: int = 5
    k_shot: int = 5
    q_query: int = 5
    eval_episodes: int = 300

    split_seed: int = 42
    seeds: Tuple[int, ...] = (42, 123, 456)

    use_projection: bool = False
    embed_dim: int = 256
    normalize_embeddings: bool = False
    squared_distance: bool = True

    mapping_version: str = "pd_mapping_v1"
    eval_seed: int = 20260725

    direct_checkpoint_paths: Tuple[str, ...] = (
        "/kaggle/input/models/tanvirakon/bse-42/pytorch/default/1/vanilla_baseline_seed42.pth",
    )

CFG = Config()

MODE_SEED_OFFSET = {
    "ALL": 1,
    "SEEN": 2,
    "VAL": 3,
    "UNSEEN": 4
}

# ============================================================
# Seed & Reproducibility
# ============================================================

def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def worker_init_fn(worker_id: int):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def autocast_context(device: str):
    if str(device).startswith("cuda"):
        try:
            return torch.amp.autocast(device_type="cuda")
        except AttributeError:
            return torch.cuda.amp.autocast()
    return contextlib.nullcontext()

# ============================================================
# Vanilla Baseline Model
# ============================================================

class BaselineEvalModel(nn.Module):
    def __init__(
        self,
        pretrained: bool = False,
        use_projection: bool = False,
        embed_dim: int = 256,
        normalize_embeddings: bool = False,
    ):
        super().__init__()

        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        effnet = models.efficientnet_b0(weights=weights)

        self.features = effnet.features
        self.pooling = nn.AdaptiveAvgPool2d(1)

        self.use_projection = use_projection
        self.normalize_embeddings = normalize_embeddings

        if self.use_projection:
            self.proj = nn.Linear(1280, embed_dim)

    def forward(self, x):
        x = self.features(x)
        x = self.pooling(x)
        x = torch.flatten(x, 1)

        if self.use_projection:
            x = self.proj(x)

        if self.normalize_embeddings:
            x = F.normalize(x, dim=1)

        return x

# ============================================================
# Safe Checkpoint Finder
# ============================================================

def find_checkpoint(run_seed: int, cfg: Config) -> Optional[str]:
    candidate_names = [
        f"baseline_seed_{run_seed}.pth",
        f"baseline_seed_{run_seed}.pt",
        f"baseline_seed_{run_seed}.ckpt",
        f"vanilla_baseline_seed_{run_seed}.pth",
        f"vanilla_baseline_seed{run_seed}.pth",
        f"seed_{run_seed}.pth",
        f"seed{run_seed}.pth",
    ]

    if run_seed == 42:
        candidate_names += [
            "seed42.pth",
            "baseline_seed42.pth",
            "vanilla_baseline_seed42.pth",
            "vill_baseline.pth",
            "vill_baseline_tst_score.pth",
        ]

    seen = set()
    unique_names = []
    for name in candidate_names:
        if name not in seen:
            seen.add(name)
            unique_names.append(name)
    candidate_names = unique_names
    candidate_set = set(candidate_names)

    for path in cfg.direct_checkpoint_paths:
        if os.path.isfile(path):
            fname = os.path.basename(path).lower()
            if f"seed_{run_seed}" in fname or f"seed{run_seed}" in fname or (run_seed == 42 and "42" in fname):
                return path

    search_bases = [".", cfg.results_dir, "/kaggle/working"]
    for base in search_bases:
        if not base: continue
        for name in candidate_names:
            path = os.path.join(base, name)
            if os.path.isfile(path):
                return path

    base_input = "/kaggle/input"
    if os.path.isdir(base_input):
        prune_tokens = (
            "plantvillage-dataset",
            "plantdoc-test-train-merged",
            "abdallahalidev/plantvillage",
            "tanvirakon/plantdoc",
            "color",
        )

        for root, dirs, files in os.walk(base_input):
            root_lower = root.lower()
            if any(token in root_lower for token in prune_tokens):
                dirs[:] = []
                continue
            
            dirs[:] = [d for d in dirs if not d.startswith(".")]

            for file in files:
                if file in candidate_set:
                    full_path = os.path.join(root, file)
                    if os.path.isfile(full_path):
                        return full_path

    return None

def load_checkpoint_compat(model, path: str, device: str):
    try:
        state = torch.load(path, map_location=device, weights_only=True)
    except Exception:
        state = torch.load(path, map_location=device)

    if isinstance(state, nn.Module):
        state = state.state_dict()

    if isinstance(state, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            if key in state and isinstance(state[key], dict):
                state = state[key]
                break

        state = {
            k.replace("module.", ""): v
            for k, v in state.items()
        }

    missing, unexpected = model.load_state_dict(state, strict=False)

    if missing:
        raise RuntimeError(
            f"Checkpoint is missing required keys for this vanilla baseline:\n{missing}"
        )

    if unexpected:
        print(f"WARNING: unexpected checkpoint keys were ignored: {unexpected}")

# ============================================================
# Exact UA-ProtoNet Episode Generation / Loading
# ============================================================

def pairs_digest(pairs: List[Tuple[str, int, int]]) -> str:
    items = sorted([(str(name), int(pv), int(pd)) for name, pv, pd in pairs], key=lambda x: x[0])
    s = "|".join(f"{name}:{pv}:{pd}" for name, pv, pd in items)
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:10]

def generate_cross_domain_episodes(
    pairs: List[Tuple[str, int, int]],
    pv_class_to_indices: Dict[int, List[int]],
    pd_class_to_indices: Dict[int, List[int]],
    n_way: int,
    k_shot: int,
    q_query: int,
    episodes: int,
    rng: random.Random
) -> List[dict]:
    episode_list = []

    for episode_id in range(episodes):
        sampled_pairs = rng.sample(pairs, n_way)

        support_indices = []
        support_labels = []
        query_indices = []
        query_labels = []

        for mapped_label, (cname, pv_idx, pd_idx) in enumerate(sampled_pairs):
            pv_avail = pv_class_to_indices[pv_idx]
            pd_avail = pd_class_to_indices[pd_idx]

            selected_pv = rng.sample(pv_avail, k_shot)
            selected_pd = rng.sample(pd_avail, q_query)

            support_indices.extend(selected_pv)
            support_labels.extend([mapped_label] * k_shot)

            query_indices.extend(selected_pd)
            query_labels.extend([mapped_label] * q_query)

        episode_list.append({
            "episode_id": episode_id,
            "pairs": sampled_pairs,
            "support_indices": support_indices,
            "support_labels": support_labels,
            "query_indices": query_indices,
            "query_labels": query_labels
        })

    return episode_list

def load_or_create_cross_domain_episodes(
    split_name: str,
    pairs: List[Tuple[str, int, int]],
    pv_class_to_indices: Dict[int, List[int]],
    pd_class_to_indices: Dict[int, List[int]],
    episodes: int,
    rng_seed: int,
    cfg: Config
) -> List[dict]:
    os.makedirs(cfg.episode_dir, exist_ok=True)

    digest = pairs_digest(pairs)

    filename = os.path.join(
        cfg.episode_dir,
        f"pd_cross_{split_name}_{cfg.mapping_version}_pairs{digest}_rng{rng_seed}_"
        f"split{cfg.split_seed}_n{cfg.n_way}_k{cfg.k_shot}_q{cfg.q_query}_ep{episodes}.json"
    )

    if os.path.exists(filename):
        print(f"      Loading fixed PlantDoc {split_name} episodes from: {filename}")
        with open(filename, "r") as f:
            return json.load(f)

    print(f"      Creating fixed PlantDoc {split_name} episodes at: {filename}")
    rng = random.Random(rng_seed)

    episode_list = generate_cross_domain_episodes(
        pairs=pairs,
        pv_class_to_indices=pv_class_to_indices,
        pd_class_to_indices=pd_class_to_indices,
        n_way=cfg.n_way,
        k_shot=cfg.k_shot,
        q_query=cfg.q_query,
        episodes=episodes,
        rng=rng
    )

    with open(filename, "w") as f:
        json.dump(episode_list, f)

    return episode_list

# ============================================================
# Cross-Domain Episodic Dataset
# ============================================================

class FixedCrossDomainDataset(Dataset):
    def __init__(self, pv_dataset, pd_dataset, episodes: List[dict]):
        self.pv_dataset = pv_dataset
        self.pd_dataset = pd_dataset
        self.episodes = episodes

    def __len__(self):
        return len(self.episodes)

    def __getitem__(self, idx):
        ep = self.episodes[idx]

        support_x = torch.stack([
            self.pv_dataset[int(i)][0] for i in ep["support_indices"]
        ])
        query_x = torch.stack([
            self.pd_dataset[int(i)][0] for i in ep["query_indices"]
        ])

        support_y = torch.tensor(ep["support_labels"], dtype=torch.long)
        query_y = torch.tensor(ep["query_labels"], dtype=torch.long)

        return support_x, support_y, query_x, query_y

# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate_plantdoc_baseline_mode(
    model,
    test_loader,
    cfg: Config,
    run_seed: int,
    mode_name: str
) -> float:
    model.eval()

    records = []
    total_episodes = len(test_loader)

    print(f"      [{mode_name}] Evaluating {total_episodes} fixed UA episodes")

    for episode_idx, (support_x, support_y, query_x, query_y) in enumerate(test_loader):
        support_x = support_x.squeeze(0).to(cfg.device, non_blocking=True)
        support_y = support_y.squeeze(0).to(cfg.device, non_blocking=True)
        query_x = query_x.squeeze(0).to(cfg.device, non_blocking=True)
        query_y = query_y.squeeze(0).to(cfg.device, non_blocking=True)

        with autocast_context(cfg.device):
            all_emb = model(torch.cat([support_x, query_x], dim=0))

        all_emb = all_emb.float()

        n_support = support_x.size(0)
        support_emb = all_emb[:n_support]
        query_emb = all_emb[n_support:]

        prototypes = torch.stack([
            support_emb[support_y == c].mean(dim=0)
            for c in range(cfg.n_way)
        ])

        dist = torch.cdist(query_emb, prototypes, p=2)

        if cfg.squared_distance:
            dist = dist.pow(2)

        logits = -dist

        acc = (logits.argmax(dim=1) == query_y).float().mean().item()

        records.append({
            "seed": run_seed,
            "mode": mode_name,
            "episode_id": episode_idx,
            "det_acc": acc
        })

        if (episode_idx + 1) % 50 == 0 or (episode_idx + 1) == total_episodes:
            print(f"      [{mode_name}] Episode {episode_idx + 1}/{total_episodes}")

    df = pd.DataFrame(records)

    os.makedirs(cfg.results_dir, exist_ok=True)
    csv_path = os.path.join(
        cfg.results_dir,
        f"baseline_pd_{mode_name}_episode_metrics_seed_{run_seed}.csv"
    )

    df.to_csv(csv_path, index=False)

    mean_acc = float(df["det_acc"].mean())
    print(f"      [{mode_name}] Baseline mean acc: {mean_acc * 100:.2f}%")

    return mean_acc

# ============================================================
# Student's t Confidence Interval
# ============================================================

def compute_student_t_ci(values: List[float]) -> dict:
    vals = np.asarray(values, dtype=np.float64)
    n = len(vals)

    if n == 0:
        return {"mean": float("nan"), "ci": float("nan"), "std": float("nan"), "tcrit": float("nan"), "df": 0}

    mean = float(np.mean(vals))

    if n == 1:
        return {"mean": mean, "ci": 0.0, "std": 0.0, "tcrit": float("nan"), "df": 0}

    std = float(np.std(vals, ddof=1))
    df = n - 1

    tcrit = float(t.ppf(0.975, df))
    ci = tcrit * (std / math.sqrt(n))

    return {"mean": mean, "ci": ci, "std": std, "tcrit": tcrit, "df": df}

# ============================================================
# Summary Reporting
# ============================================================

def summarize_baseline_pd(all_metrics: Dict[str, List[dict]], cfg: Config):
    os.makedirs(cfg.results_dir, exist_ok=True)

    seed_csv = os.path.join(cfg.results_dir, "baseline_pd_phase1_seed_metrics.csv")

    seed_rows = []

    mode_order = ["ALL", "SEEN", "VAL", "UNSEEN"]
    modes = mode_order + [m for m in all_metrics.keys() if m not in mode_order]

    for mode in modes:
        for m in all_metrics.get(mode, []):
            seed_rows.append({
                "mode": mode,
                "seed": m["seed"],
                "det_test_acc_pct": m["det_acc"] * 100.0
            })

    pd.DataFrame(seed_rows).to_csv(seed_csv, index=False)
    print(f"Saved baseline seed-level metrics: {seed_csv}")

    summary_csv = os.path.join(cfg.results_dir, "baseline_pd_phase1_summary.csv")

    summary_rows = []

    print("\n" + "=" * 95)
    print("  BASELINE CROSS-DOMAIN SUMMARY: PlantDoc Evaluation")
    print("=" * 95)

    for mode in modes:
        metrics_list = all_metrics.get(mode, [])
        if not metrics_list:
            continue

        print(f"  --- {mode} SPLIT ---")

        for m in metrics_list:
            print(f"    Seed {m['seed']:5d} | Det: {m['det_acc'] * 100:.2f}%")

        vals = [m["det_acc"] * 100.0 for m in metrics_list]
        stats = compute_student_t_ci(vals)

        summary_rows.append({
            "mode": mode,
            "metric": "Det Acc (%)",
            "mean": stats["mean"],
            "student_t_95_ci_half_width": stats["ci"],
            "std": stats["std"],
            "df": stats["df"],
            "tcrit": stats["tcrit"],
            "seed_values": ";".join(f"{v:.6f}" for v in vals)
        })

        print(
            f"    Agg Det Acc (%): "
            f"{stats['mean']:.2f} ± {stats['ci']:.2f} "
            f"(SD={stats['std']:.2f}, df={stats['df']}, tcrit={stats['tcrit']:.3f})"
        )
        print()

    pd.DataFrame(summary_rows).to_csv(summary_csv, index=False)
    print(f"Saved baseline summary metrics: {summary_csv}")
    print("=" * 95)

# ============================================================
# Main Execution
# ============================================================

if __name__ == "__main__":
    print("=" * 95)
    print("  Vanilla ProtoNet Baseline: Multi-Seed PlantDoc Evaluation")
    print(f"  Seeds: {CFG.seeds}")
    print(f"  Protocol: {CFG.n_way}-way {CFG.k_shot}-shot {CFG.q_query}-query")
    print(f"  Episodes per mode: {CFG.eval_episodes}")
    print("=" * 95)

    os.makedirs(CFG.results_dir, exist_ok=True)

    with open(os.path.join(CFG.results_dir, "baseline_pd_eval_config.json"), "w") as f:
        json.dump({k: str(v) if isinstance(v, tuple) else v for k, v in vars(CFG).items()}, f, indent=2)

    test_tfms = transforms.Compose([
        transforms.Resize((CFG.image_size, CFG.image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print("Loading datasets...")

    pv_ds = datasets.ImageFolder(CFG.plantvillage_root, transform=test_tfms)
    pd_ds = datasets.ImageFolder(CFG.plantdoc_root, transform=test_tfms)

    pv_class_to_indices = defaultdict(list)
    for idx, target in enumerate(pv_ds.targets):
        pv_class_to_indices[target].append(idx)
    pv_class_to_indices = dict(pv_class_to_indices)

    pd_class_to_indices = defaultdict(list)
    for idx, target in enumerate(pd_ds.targets):
        pd_class_to_indices[target].append(idx)
    pd_class_to_indices = dict(pd_class_to_indices)

    pv_name_to_idx = {c: i for i, c in enumerate(pv_ds.classes)}
    pd_name_to_idx = {c: i for i, c in enumerate(pd_ds.classes)}

    PD_TO_PV_MAPPING = {
        "Apple_Scab_Leaf":              "Apple___Apple_scab",
        "Apple_rust_leaf":              "Apple___Cedar_apple_rust",
        "Apple_leaf":                   "Apple___healthy",
        "Bell_pepper_leaf":             "Pepper,_bell___healthy",
        "Bell_pepper_leaf_spot":        "Pepper,_bell___Bacterial_spot",
        "Blueberry_leaf":               "Blueberry___healthy",
        "Cherry_leaf":                  "Cherry_(including_sour)___healthy",
        "Corn_Gray_leaf_spot":          "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
        "Corn_leaf_blight":             "Corn_(maize)___Northern_Leaf_Blight",
        "Corn_rust_leaf":               "Corn_(maize)___Common_rust_",
        "grape_leaf":                   "Grape___healthy",
        "grape_leaf_black_rot":         "Grape___Black_rot",
        "Peach_leaf":                   "Peach___healthy",
        "Potato_leaf_early_blight":     "Potato___Early_blight",
        "Potato_leaf_late_blight":      "Potato___Late_blight",
        "Raspberry_leaf":               "Raspberry___healthy",
        "Soyabean_leaf":                "Soybean___healthy",
        "Squash_Powdery_mildew_leaf":   "Squash___Powdery_mildew",
        "Strawberry_leaf":              "Strawberry___healthy",
        "Tomato_Early_blight_leaf":     "Tomato___Early_blight",
        "Tomato_Septoria_leaf_spot":    "Tomato___Septoria_leaf_spot",
        "Tomato_leaf":                  "Tomato___healthy",
        "Tomato_leaf_bacterial_spot":   "Tomato___Bacterial_spot",
        "Tomato_leaf_late_blight":      "Tomato___Late_blight",
        "Tomato_leaf_mosaic_virus":     "Tomato___Tomato_mosaic_virus",
        "Tomato_leaf_yellow_virus":     "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
        "Tomato_mold_leaf":             "Tomato___Leaf_Mold",
    }

    usable_common_pairs_raw = []

    for pd_name, pv_name in PD_TO_PV_MAPPING.items():
        if pd_name not in pd_name_to_idx or pv_name not in pv_name_to_idx:
            continue

        pv_idx = pv_name_to_idx[pv_name]
        pd_idx = pd_name_to_idx[pd_name]

        if len(pv_class_to_indices[pv_idx]) >= CFG.k_shot and len(pd_class_to_indices[pd_idx]) >= CFG.q_query:
            usable_common_pairs_raw.append((pd_name, pv_idx, pd_idx))

    if len(pv_ds.classes) < 38:
        raise ValueError(f"Expected at least 38 PlantVillage classes, found {len(pv_ds.classes)}.")

    pv_all = list(range(len(pv_ds.classes)))
    random.Random(CFG.split_seed).shuffle(pv_all)

    train_classes = pv_all[:23]
    val_classes = pv_all[23:31]
    test_classes = pv_all[31:38]

    train_set = set(train_classes)
    val_set = set(val_classes)
    test_set = set(test_classes)
    split_class_set = train_set | val_set | test_set

    usable_common_pairs = sorted([p for p in usable_common_pairs_raw if p[1] in split_class_set], key=lambda x: x[0])

    seen_pairs = sorted([p for p in usable_common_pairs if p[1] in train_set], key=lambda x: x[0])
    val_pairs = sorted([p for p in usable_common_pairs if p[1] in val_set], key=lambda x: x[0])
    unseen_pairs = sorted([p for p in usable_common_pairs if p[1] in test_set], key=lambda x: x[0])
    all_pairs = sorted(usable_common_pairs, key=lambda x: x[0])

    print(f"  ALL pairs:    {len(all_pairs)}")
    print(f"  SEEN pairs:   {len(seen_pairs)}")
    print(f"  VAL pairs:    {len(val_pairs)}")
    print(f"  UNSEEN pairs: {len(unseen_pairs)}")

    mode_configs = {
        "ALL":    (all_pairs,    CFG.split_seed + MODE_SEED_OFFSET["ALL"] * 100000),
        "SEEN":   (seen_pairs,   CFG.split_seed + MODE_SEED_OFFSET["SEEN"] * 100000),
        "VAL":    (val_pairs,    CFG.split_seed + MODE_SEED_OFFSET["VAL"] * 100000),
        "UNSEEN": (unseen_pairs, CFG.split_seed + MODE_SEED_OFFSET["UNSEEN"] * 100000),
    }

    mode_loaders = {}
    use_cuda = str(CFG.device).startswith("cuda")

    loader_kwargs = dict(
        batch_size=CFG.batch_size,
        num_workers=CFG.num_workers,
        pin_memory=use_cuda,
        persistent_workers=False,
        worker_init_fn=worker_init_fn
    )

    for mode, (pairs, rng_seed) in mode_configs.items():
        if len(pairs) < CFG.n_way:
            print(f"Skipping {mode} evaluation: not enough classes ({len(pairs)} < {CFG.n_way})")
            continue

        episodes_json = load_or_create_cross_domain_episodes(
            split_name=mode,
            pairs=pairs,
            pv_class_to_indices=pv_class_to_indices,
            pd_class_to_indices=pd_class_to_indices,
            episodes=CFG.eval_episodes,
            rng_seed=rng_seed,
            cfg=CFG
        )

        ds = FixedCrossDomainDataset(
            pv_dataset=pv_ds,
            pd_dataset=pd_ds,
            episodes=episodes_json
        )

        mode_loaders[mode] = DataLoader(ds, shuffle=False, **loader_kwargs)

    all_seed_metrics = defaultdict(list)

    for seed_idx, run_seed in enumerate(CFG.seeds):
        print("\n" + "-" * 95)
        print(f"   EVALUATING BASELINE SEED {run_seed} ({seed_idx + 1}/{len(CFG.seeds)})")
        print("-" * 95)

        checkpoint_path = find_checkpoint(run_seed, CFG)

        if checkpoint_path is None:
            print(f"WARNING: Could not find baseline checkpoint for seed {run_seed}. Skipping.")
            continue

        print(f"   Loading weights from: {checkpoint_path}")

        model = BaselineEvalModel(
            pretrained=False,
            use_projection=CFG.use_projection,
            embed_dim=CFG.embed_dim,
            normalize_embeddings=CFG.normalize_embeddings
        ).to(CFG.device)

        load_checkpoint_compat(model, checkpoint_path, CFG.device)

        set_seed(CFG.eval_seed)

        for mode, loader in mode_loaders.items():
            print(f"\n   Evaluating {mode} split")

            acc = evaluate_plantdoc_baseline_mode(
                model=model,
                test_loader=loader,
                cfg=CFG,
                run_seed=run_seed,
                mode_name=mode
            )

            all_seed_metrics[mode].append({
                "seed": run_seed,
                "det_acc": acc
            })

        del model
        if use_cuda:
            torch.cuda.empty_cache()
        gc.collect()

    summarize_baseline_pd(all_seed_metrics, CFG)

  Vanilla ProtoNet Baseline: Multi-Seed PlantDoc Evaluation
  Seeds: (42, 123, 456)
  Protocol: 5-way 5-shot 5-query
  Episodes per mode: 300
Loading datasets...
  ALL pairs:    27
  SEEN pairs:   15
  VAL pairs:    7
  UNSEEN pairs: 5
      Creating fixed PlantDoc ALL episodes at: ./pd_cross_ALL_pd_mapping_v1_pairse90005f6eb_rng100042_split42_n5_k5_q5_ep300.json
      Creating fixed PlantDoc SEEN episodes at: ./pd_cross_SEEN_pd_mapping_v1_pairs56e21b1e78_rng200042_split42_n5_k5_q5_ep300.json
      Creating fixed PlantDoc VAL episodes at: ./pd_cross_VAL_pd_mapping_v1_pairsd249e9cf71_rng300042_split42_n5_k5_q5_ep300.json
      Creating fixed PlantDoc UNSEEN episodes at: ./pd_cross_UNSEEN_pd_mapping_v1_pairsb8e63f6b17_rng400042_split42_n5_k5_q5_ep300.json

-----------------------------------------------------------------------------------------------
   EVALUATING BASELINE SEED 42 (1/3)
-----------------------------------------------------------------------------------------------
   Loa

# c